In [47]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import random

In [48]:
season_features_m = pd.read_csv('season_features_m.csv')
season_features_w = pd.read_csv('season_features_w.csv')

In [49]:

seeds_m = pd.read_csv('useful_data/MNCAATourneySeeds.csv')
seeds_w = pd.read_csv('useful_data/WNCAATourneySeeds.csv')

tourney_m = pd.read_csv('useful_data/MNCAATourneyDetailedResults.csv')
tourney_w = pd.read_csv('useful_data/WNCAATourneyDetailedResults.csv')


In [50]:


random.seed(52)


temp_m = tourney_m[["Season", "WTeamID","LTeamID"]].copy()
for i in range(len(temp_m)):
    pair = random.choice([("WTeamID","LTeamID"), ("LTeamID","WTeamID")])
    pair = list(pair)
    temp_m.loc[i,["WTeamID","LTeamID"]] = temp_m.loc[i,pair].values

temp_w = tourney_w[["Season", "WTeamID","LTeamID"]].copy()
for i in range(len(temp_w)):
    pair = random.choice([("WTeamID","LTeamID"), ("LTeamID","WTeamID")])
    pair = list(pair)
    temp_w.loc[i,["WTeamID","LTeamID"]] = temp_w.loc[i,pair].values



In [51]:


# zakładam, że masz już dwa DataFrame: season_features_m i m_seeds

merged_m = season_features_m.merge(
    seeds_m,
    how='left',          # left join
    on=['Season', 'TeamID']  # kolumny, po których łączymy
)

merged_w = season_features_w.merge(
    seeds_w,
    how='left',          # left join
    on=['Season', 'TeamID']  # kolumny, po których łączymy
)

In [52]:
# merge dla zwycięzców (WTeamID)
m_tourney_with_win = temp_m.merge(
    merged_m,
    how='left',
    left_on=['Season', 'WTeamID'],
    right_on=['Season', 'TeamID'],
    suffixes=('', '_1')  # dodajemy "_w" do kolumn z merged, aby rozróżnić
)

# merge dla przegranych (LTeamID)
m_tourney_full = m_tourney_with_win.merge(
    merged_m,
    how='left',
    left_on=['Season', 'LTeamID'],
    right_on=['Season', 'TeamID'],
    suffixes=('', '_2')  # dodajemy "_l" do kolumn z merged, aby rozróżnić
)

In [53]:
# merge dla zwycięzców (WTeamID)
w_tourney_with_win = temp_w.merge(
    merged_w,
    how='left',
    left_on=['Season', 'WTeamID'],
    right_on=['Season', 'TeamID'],
    suffixes=('', '_1')  # dodajemy "_w" do kolumn z merged, aby rozróżnić
)

# merge dla przegranych (LTeamID)
w_tourney_full = w_tourney_with_win.merge(
    merged_w,
    how='left',
    left_on=['Season', 'LTeamID'],
    right_on=['Season', 'TeamID'],
    suffixes=('', '_2')  # dodajemy "_l" do kolumn z merged, aby rozróżnić
)

In [54]:

m_tourney_full['Is1Winner'] = (m_tourney_full['WTeamID'] == tourney_m['WTeamID']).astype(int)
w_tourney_full['Is1Winner'] = (w_tourney_full['WTeamID'] == tourney_w['WTeamID']).astype(int)
m_tourney_full['1TeamID'] = m_tourney_full['WTeamID']
m_tourney_full['2TeamID'] = m_tourney_full['LTeamID']
w_tourney_full['1TeamID'] = w_tourney_full['WTeamID']
w_tourney_full['2TeamID'] = w_tourney_full['LTeamID']

m_tourney_full.drop(['WTeamID', 'LTeamID', 'TeamID'], axis=1, inplace=True)
w_tourney_full.drop(['WTeamID', 'LTeamID', 'TeamID'], axis=1,inplace=True)

In [60]:
for col in ['Seed', 'Seed_2']:
    m_tourney_full[col] = (
        m_tourney_full[col]
        .str.extract(r'(\d+)')      # wyciąga cyfry
        .astype(float)              # żeby NaN działał
        .fillna(17)                 # brak seed -> 17
        .astype(int)
    )
    w_tourney_full[col] = (
        w_tourney_full[col]
        .str.extract(r'(\d+)')      # wyciąga cyfry
        .astype(float)              # żeby NaN działał
        .fillna(17)                 # brak seed -> 17
        .astype(int)
    )

    # normalizacja + odwrócenie
    m_tourney_full[col] = (17 - m_tourney_full[col]) / 16
    w_tourney_full[col] = (17 - w_tourney_full[col]) / 16

In [62]:
# zapis do pliku CSV
m_tourney_full.to_csv("m_tourney_full.csv", index=False)
w_tourney_full.to_csv("w_tourney_full.csv", index=False)

In [63]:
# Train: sezony do 2023 włącznie
train_m = m_tourney_full[m_tourney_full['Season'] <= 2023]

# Test: sezon 2024
test_m = m_tourney_full[m_tourney_full['Season'] == 2024]

# Validation: sezon 2025
val_m = m_tourney_full[m_tourney_full['Season'] == 2025]

# opcjonalnie podgląd liczby wier

In [64]:

train_m.to_csv("tourney_train_m.csv", index=False)

test_m.to_csv("tourney_test_m.csv", index=False)

val_m.to_csv("tourney_val_m.csv", index=False)


In [65]:
# Train: sezony do 2023 włącznie
train_w = w_tourney_full[w_tourney_full['Season'] <= 2023]

# Test: sezon 2024
test_w = w_tourney_full[w_tourney_full['Season'] == 2024]

# Validation: sezon 2025
val_w = w_tourney_full[w_tourney_full['Season'] == 2025]

# opcjonalnie podgląd liczby wier

train_w.to_csv("tourney_train_w.csv", index=False)

test_w.to_csv("tourney_test_w.csv", index=False)

val_w.to_csv("tourney_val_w.csv", index=False)
